# DISCRETIO: Box-Scale Counterfactual Intervention on IDD-PeD

**Task 1 Benchmark**: Probing Latent Behavioral Priors in Trajectory Prediction Models (BiTraP, SGNet, Kinematic Baselines)

### The Scientific Question
When a world/trajectory model forecasts pedestrian futures, does altering the bounding box scale around the **ground contact point** (foot-invariant) reveal a **latent behavioral prior** (e.g. child hesitation, slower crossing speed)? Or does the model simply misinterpret a smaller box as being *farther away* (perspective depth confusion)?

---

## 1. Environment Setup & Download IDD-PeD Trajectory Cache
We download the annotations, precomputed trajectory cache (`data_cache/iddp_database.pkl`), and checkpoints from the Hugging Face repository (~1GB instead of 100GB of video).

In [ ]:
# Install lightweight dependencies
!pip install -q huggingface_hub pyyaml easydict scipy matplotlib

import os
from huggingface_hub import snapshot_download

print('Downloading IDD-PeD dataset cache and checkpoints from Hugging Face...')
data_dir = snapshot_download(
    repo_id='Ruthvik9/IDD-PeD',
    repo_type='dataset',
    allow_patterns=['data_cache/*', 'annotations/*', 'checkpoints/*'],
    local_dir='iddped_data'
)
print(f'Files ready at: {data_dir}')

## 2. Ground-Anchored Box-Scale Intervention Implementation
To simulate a child, we scale height and width by $\alpha = 0.65$.
**Crucial condition**: The bottom coordinate $y_2$ and horizontal center $x_{\text{foot}}$ remain **strictly invariant** to preserve perspective ground contact.

In [ ]:
import numpy as np

def apply_ground_anchored_scale(bboxes, scale=0.65):
    """
    Scales bounding box height and width while anchoring strictly to the ground point (y2).
    bboxes shape: (T, 4) in [x1, y1, x2, y2]
    """
    boxes = np.asarray(bboxes, dtype=np.float32)
    x1, y1, x2, y2 = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]
    w, h = x2 - x1, y2 - y1
    x_foot = (x1 + x2) / 2.0
    
    w_new = w * scale
    h_new = h * scale
    
    new_y2 = y2.copy()           # Foot contact stays identical
    new_y1 = y2 - h_new          # Top shifts down toward feet
    new_x1 = x_foot - w_new / 2.0
    new_x2 = x_foot + w_new / 2.0
    return np.stack([new_x1, new_y1, new_x2, new_y2], axis=1)

def apply_centroid_scale(bboxes, scale=0.65):
    """
    Negative Control: Shrinks around center (shifts y2 upward, simulating greater depth).
    """
    boxes = np.asarray(bboxes, dtype=np.float32)
    x1, y1, x2, y2 = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]
    w, h = x2 - x1, y2 - y1
    xc, yc = (x1 + x2) / 2.0, (y1 + y2) / 2.0
    
    return np.stack([
        xc - (w * scale) / 2.0,
        yc - (h * scale) / 2.0,
        xc + (w * scale) / 2.0,
        yc + (h * scale) / 2.0
    ], axis=1)

## 3. Visual Verification: Perspective & Ground-Contact Invariance
We plot the adult box vs. the ground-anchored child box vs. the centroid-scaled box.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

adult_box = np.array([[500, 300, 560, 480]], dtype=np.float32)
child_ground = apply_ground_anchored_scale(adult_box, scale=0.65)
child_centroid = apply_centroid_scale(adult_box, scale=0.65)

fig, ax = plt.subplots(figsize=(8, 6))
ax.set_xlim(450, 600)
ax.set_ylim(520, 250)  # Inverted for camera image plane

# Road ground plane line
ax.axhline(480, color='black', linestyle='--', label='Road Ground Plane (y2 = 480)')

# Adult
b = adult_box[0]
ax.add_patch(patches.Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1], fill=None, edgecolor='blue', lw=2, label='Adult (Original)'))

# Ground-anchored child
b_g = child_ground[0]
ax.add_patch(patches.Rectangle((b_g[0], b_g[1]), b_g[2]-b_g[0], b_g[3]-b_g[1], fill=None, edgecolor='green', lw=2, label='Child (Ground-Anchored)'))

# Centroid scaled
b_c = child_centroid[0]
ax.add_patch(patches.Rectangle((b_c[0], b_c[1]), b_c[2]-b_c[0], b_c[3]-b_c[1], fill=None, edgecolor='red', linestyle=':', lw=2, label='Centroid (False Depth Shift)'))

ax.legend(loc='upper right')
ax.set_title('Geometric Verification: Ground-Contact Invariance')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Original y2 (feet): {adult_box[0, 3]}")
print(f"Ground-anchored y2: {child_ground[0, 3]} (Diff: {child_ground[0, 3] - adult_box[0, 3]} px)")
print(f"Centroid-scaled y2: {child_centroid[0, 3]} (Diff: {child_centroid[0, 3] - adult_box[0, 3]} px - False Depth Shift!)")

## 4. Robust IDD-PeD Trajectory Parser
In IDD-PeD / PIE, the database dictionary is structured hierarchically:
`database[video_id]['ped_annotations'][ped_id]['bbox']`

We provide `extract_all_pedestrians()` which automatically traverses the database schema and extracts all pedestrian bounding box tracks.

In [ ]:
import pickle

def extract_all_pedestrians(raw_db):
    """
    Robustly extracts pedestrian tracks from IDD-PeD database.
    Handles nested schema: raw_db[video_id]['ped_annotations'][ped_id]['bbox']
    """
    tracks = {}
    if not isinstance(raw_db, dict):
        return tracks
        
    for vid_id, vid_info in raw_db.items():
        if not isinstance(vid_info, dict):
            continue
            
        # Schema 1: Standard IDD-PeD / PIE nested structure
        if 'ped_annotations' in vid_info and isinstance(vid_info['ped_annotations'], dict):
            for ped_id, ped_data in vid_info['ped_annotations'].items():
                if isinstance(ped_data, dict) and 'bbox' in ped_data and len(ped_data['bbox']) > 0:
                    tracks[f'{vid_id}_{ped_id}'] = np.array(ped_data['bbox'], dtype=np.float32)
        # Schema 2: Flat dictionary
        elif 'bbox' in vid_info and len(vid_info['bbox']) > 0:
            tracks[str(vid_id)] = np.array(vid_info['bbox'], dtype=np.float32)
        # Schema 3: Set -> Video -> Ped
        else:
            for sub_k, sub_v in vid_info.items():
                if isinstance(sub_v, dict) and 'ped_annotations' in sub_v:
                    for ped_id, ped_data in sub_v['ped_annotations'].items():
                        if isinstance(ped_data, dict) and 'bbox' in ped_data and len(ped_data['bbox']) > 0:
                            tracks[f'{vid_id}_{sub_k}_{ped_id}'] = np.array(ped_data['bbox'], dtype=np.float32)
    return tracks

# Load real IDD-PeD database
db_path = 'iddped_data/data_cache/iddp_database.pkl'
if os.path.exists(db_path):
    with open(db_path, 'rb') as f:
        raw_database = pickle.load(f)
    all_tracks = extract_all_pedestrians(raw_database)
    print(f'Successfully loaded {len(raw_database)} video clips and extracted {len(all_tracks)} pedestrian tracks!')
else:
    print('Data cache not found locally; generating simulated IDD-PeD tracks for demonstration...')
    all_tracks = {
        f'sim_ped_{i}': np.array([[400 + i*5 + t*2, 300, 440 + i*5 + t*2, 450] for t in range(45)], dtype=np.float32)
        for i in range(30)
    }
    print(f'Ready with {len(all_tracks)} benchmark tracks.')

# Pick valid track with length >= 45 frames (15 obs + 30 pred)
valid_keys = [k for k, v in all_tracks.items() if len(v) >= 45]
if not valid_keys:
    valid_keys = list(all_tracks.keys())

selected_key = valid_keys[0]
raw_boxes = all_tracks[selected_key]
print(f'Selected Track ID: {selected_key} with {len(raw_boxes)} frames.')
print(f'First frame bounding box [x1, y1, x2, y2]: {raw_boxes[0]}')

## 5. Diagnostic Rollout Evaluation Engine
Computes:
1. **Trajectory Divergence ($D_\tau$):** Mean Euclidean distance between rollouts.
2. **Speed Ratio ($v_{\text{child}} / v_{\text{adult}}$):** Detects hesitation or deceleration ($< 0.90$).
3. **Foot Y-Shift ($\Delta y_{\text{foot}}$):** Perspective depth check (values $< -5\text{ px}$ indicate false distance confusion).

In [ ]:
def evaluate_diagnostics(pred_orig, pred_child, fps=30.0):
    # Foot points
    foot_orig = np.stack([(pred_orig[:, 0] + pred_orig[:, 2]) / 2.0, pred_orig[:, 3]], axis=-1)
    foot_child = np.stack([(pred_child[:, 0] + pred_child[:, 2]) / 2.0, pred_child[:, 3]], axis=-1)
    
    # 1. Trajectory Divergence (D_tau)
    d_tau = float(np.mean(np.linalg.norm(foot_orig - foot_child, axis=-1)))
    
    # 2. Speed Shift / Hesitation Ratio
    dt = 1.0 / fps
    vel_orig = np.diff(foot_orig, axis=0) / dt
    vel_child = np.diff(foot_child, axis=0) / dt
    
    speed_orig = float(np.mean(np.linalg.norm(vel_orig, axis=-1))) + 1e-6
    speed_child = float(np.mean(np.linalg.norm(vel_child, axis=-1)))
    speed_ratio = speed_child / speed_orig
    
    # 3. Ground Foot Y-Shift (Depth Check)
    foot_y_shift = float(np.mean(pred_child[:, 3] - pred_orig[:, 3]))
    
    return {
        'trajectory_divergence_d_tau': round(d_tau, 3),
        'speed_ratio': round(speed_ratio, 3),
        'foot_y_shift': round(foot_y_shift, 3),
        'hesitation_detected': bool(speed_ratio < 0.90),
        'depth_confusion_detected': bool(foot_y_shift < -5.0),
        'scale_invariant': bool(d_tau < 0.5),
    }

# Slice observation window
T_obs = 15
obs_adult = raw_boxes[:T_obs]
obs_child_ground = apply_ground_anchored_scale(obs_adult, scale=0.65)
obs_child_centroid = apply_centroid_scale(obs_adult, scale=0.65)

# Trajectory Predictor Rollout Function (Kinematic baseline)
def rollout_constant_velocity(obs, horizon=30):
    w = obs[-1, 2] - obs[-1, 0]
    h = obs[-1, 3] - obs[-1, 1]
    vx = (obs[-1, 0] - obs[0, 0]) / float(len(obs))
    vy = (obs[-1, 3] - obs[0, 3]) / float(len(obs))
    
    preds = []
    cur_x, cur_y = (obs[-1, 0] + obs[-1, 2]) / 2.0, obs[-1, 3]
    for _ in range(horizon):
        cur_x += vx
        cur_y += vy
        preds.append([cur_x - w/2.0, cur_y - h, cur_x + w/2.0, cur_y])
    return np.array(preds, dtype=np.float32)

pred_adult = rollout_constant_velocity(obs_adult, horizon=30)
pred_child_ground = rollout_constant_velocity(obs_child_ground, horizon=30)
pred_child_centroid = rollout_constant_velocity(obs_child_centroid, horizon=30)

res_ground = evaluate_diagnostics(pred_adult, pred_child_ground)
res_centroid = evaluate_diagnostics(pred_adult, pred_child_centroid)

print('=== Ground-Anchored Intervention Diagnostics ===')
for k, v in res_ground.items():
    print(f'  {k}: {v}')

print('\n=== Centroid-Scaled Control Diagnostics ===')
for k, v in res_centroid.items():
    print(f'  {k}: {v}')

## 6. Plotting Trajectory Rollouts Before & After Intervention

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Extract foot trajectories
foot_obs_adult = np.stack([(obs_adult[:, 0] + obs_adult[:, 2])/2.0, obs_adult[:, 3]], axis=-1)
foot_pred_adult = np.stack([(pred_adult[:, 0] + pred_adult[:, 2])/2.0, pred_adult[:, 3]], axis=-1)
foot_pred_ground = np.stack([(pred_child_ground[:, 0] + pred_child_ground[:, 2])/2.0, pred_child_ground[:, 3]], axis=-1)
foot_pred_centroid = np.stack([(pred_child_centroid[:, 0] + pred_child_centroid[:, 2])/2.0, pred_child_centroid[:, 3]], axis=-1)

# Plot 1: 2D Motion Plane
ax1.plot(foot_obs_adult[:, 0], foot_obs_adult[:, 1], 'bo-', label='Observed (Adult)', lw=2)
ax1.plot(foot_pred_adult[:, 0], foot_pred_adult[:, 1], 'b--', label='Predicted Adult Rollout', lw=2)
ax1.plot(foot_pred_ground[:, 0], foot_pred_ground[:, 1], 'g--', label='Predicted Child (Ground-Anchored)', lw=2)
ax1.plot(foot_pred_centroid[:, 0], foot_pred_centroid[:, 1], 'r:', label='Predicted (Centroid - False Shift)', lw=2)
ax1.invert_yaxis()
ax1.set_xlabel('X Pixel Coordinate')
ax1.set_ylabel('Y Pixel Coordinate (Ground Contact)')
ax1.set_title('Pedestrian Ground Foot Path')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Box Height Profile across Time
t_steps = np.arange(len(obs_adult))
ax2.plot(t_steps, obs_adult[:, 3] - obs_adult[:, 1], 'b-', label='Adult Box Height (px)', lw=2)
ax2.plot(t_steps, obs_child_ground[:, 3] - obs_child_ground[:, 1], 'g-', label='Child Box Height (0.65x)', lw=2)
ax2.set_xlabel('Observed Frame Step')
ax2.set_ylabel('Bounding Box Height (px)')
ax2.set_title('Box Height Intervention Profile')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()